In [1]:
!pip install langchain langchain-google-genai sentence-transformers faiss-cpu


[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: C:\Users\ASUS\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [2]:
import json
import os
import requests
from typing import List, Dict
from langchain_openai import ChatOpenAI
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

In [ ]:
os.environ["OPENAI_API_KEY"] = ""
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
llm = ChatOpenAI(model="qwen/qwen-2.5-72b-instruct", temperature=0.1)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_31372\3603910320.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [4]:
class ReviewerDataLoader:
    
    def __init__(self, json_file_path: str):
        self.json_file_path = json_file_path
        self.reviewer_data = []
        
    def load_data(self, limit: int = None):
        print(f"📂 Loading reviewer data from {self.json_file_path}...")
        
        try:
            with open(self.json_file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract PR reviewer data
            pr_reviewer_data = data.get('pr_reviewer_data', [])
            
            # Apply limit only if specified
            if limit:
                self.reviewer_data = pr_reviewer_data[:limit]
                print(f"✅ Loaded {len(self.reviewer_data)} PRs with reviews (limited from {len(pr_reviewer_data)} total)")
            else:
                self.reviewer_data = pr_reviewer_data
                print(f"✅ Loaded ALL {len(self.reviewer_data)} PRs with reviews")
            
            return True
            
        except Exception as e:
            print(f"❌ Error loading data: {e}")
            return False
    
    def get_pr_summary(self, pr: Dict) -> str:
        author = pr.get('author', {}).get('username', 'Unknown') if pr.get('author') else 'Unknown'
        
        # Get repository and project info
        repo_name = pr.get('repo_name', 'Unknown Repository')
        
        # Get reviews info
        reviews = pr.get('reviews', [])
        review_summary = []
        if reviews:
            for review in reviews[:3]:  # Show first 3 reviews
                reviewer = review.get('reviewer_username', 'Unknown')
                state = review.get('state', 'unknown')
                review_summary.append(f"  - {reviewer}: {state}")
        
        # Get review comments
        review_comments = pr.get('review_comments', [])
        
        # Format created and updated dates
        created_at = pr.get('created_at', 'Unknown')
        updated_at = pr.get('updated_at', 'Unknown')
        
        summary = f"""
REPOSITORY: {repo_name}
PR #{pr.get('pr_number', 'N/A')}: {pr.get('title', 'No title')}
Author: {author}
State: {pr.get('state', 'unknown')}
Is Draft: {pr.get('is_draft', False)}
Created: {created_at}
Updated: {updated_at}

DESCRIPTION:
{pr.get('description', 'No description')}

CHANGE STATISTICS:
- Files Changed: {pr.get('changed_files_count', 0)}
- Additions: +{pr.get('additions', 0)} lines
- Deletions: -{pr.get('deletions', 0)} lines

LABELS: {', '.join(pr.get('labels', [])) if pr.get('labels') else 'None'}

REVIEWS ({len(reviews)} total):
{chr(10).join(review_summary) if review_summary else 'No reviews yet'}

REVIEW COMMENTS: {len(review_comments)} comments
"""
        return summary.strip()

# Load reviewer data
reviewer_data_file = "PR data for Reviewers\lodash_lodash_reviewer_data_test.json"

print("🚀 Loading reviewer data...")
reviewer_loader = ReviewerDataLoader(reviewer_data_file)
success = reviewer_loader.load_data()

if success:
    print(f"🎉 Successfully loaded reviewer data!")
else:
    print("❌ Failed to load reviewer data")

🚀 Loading reviewer data...
📂 Loading reviewer data from PR data for Reviewers\lodash_lodash_reviewer_data_test.json...
✅ Loaded ALL 36 PRs with reviews
🎉 Successfully loaded reviewer data!


<>:80: SyntaxWarning: invalid escape sequence '\l'
<>:80: SyntaxWarning: invalid escape sequence '\l'
C:\Users\ASUS\AppData\Local\Temp\ipykernel_31372\3391315102.py:80: SyntaxWarning: invalid escape sequence '\l'
  reviewer_data_file = "PR data for Reviewers\lodash_lodash_reviewer_data_test.json"


In [5]:
class ReviewerVectorStore:
    
    def __init__(self, embeddings_model):
        self.embeddings = embeddings_model
        self.vector_store = None
        self.documents = []
    
    def create_embeddings(self, reviewer_data: List[Dict]):
        """Convert reviewer PR data to vector embeddings"""
        print("🔄 Creating vector embeddings for reviewer data...")
        
        documents = []
        
        # Convert each PR with reviews to a document
        for pr in reviewer_data:
            # Create text content with reviewer information
            author = pr.get('author', {}).get('username', 'Unknown') if pr.get('author') else 'Unknown'
            repo_name = pr.get('repo_name', 'Unknown Repository')
            
            # Get reviewer information
            reviews = pr.get('reviews', [])
            reviewers = []
            review_states = []
            review_bodies = []
            
            for review in reviews:
                reviewers.append(review.get('reviewer_username', 'Unknown'))
                review_states.append(review.get('state', 'unknown'))
                if review.get('body'):
                    review_bodies.append(review.get('body', '')[:200])  # First 200 chars
            
            # Get review comments
            review_comments = pr.get('review_comments', [])
            comment_authors = []
            for comment in review_comments[:5]:  # First 5 comments
                if comment.get('user', {}).get('login'):
                    comment_authors.append(comment.get('user', {}).get('login'))
            
            content = f"""
Repository: {repo_name}
PR #{pr.get('pr_number')}: {pr.get('title', '')}
Author: {author}
Description: {pr.get('description', '')[:300]}...
Labels: {', '.join(pr.get('labels', []))}
State: {pr.get('state', 'unknown')}
Is Draft: {pr.get('is_draft', False)}
Files: {pr.get('changed_files_count', 0)} changed
Changes: +{pr.get('additions', 0)} -{pr.get('deletions', 0)}
Reviewers: {', '.join(reviewers)}
Review States: {', '.join(review_states)}
Review Comments Authors: {', '.join(comment_authors)}
Sample Review Content: {' | '.join(review_bodies[:2])}
"""
            
            # Create metadata
            metadata = {
                'pr_number': pr.get('pr_number'),
                'author': author,
                'state': pr.get('state'),
                'title': pr.get('title', ''),
                'repo_name': repo_name,
                'reviewers': reviewers,
                'review_count': len(reviews),
                'comment_count': len(review_comments)
            }
            
            # Create document
            doc = Document(page_content=content.strip(), metadata=metadata)
            documents.append(doc)
        
        # Create vector store
        self.vector_store = FAISS.from_documents(documents, self.embeddings)
        self.documents = documents
        
        print(f"✅ Created {len(documents)} vector embeddings for reviewer data")
        return self.vector_store
    
    def find_similar_reviewed_prs(self, query: str, k: int = 5) -> List[Document]:
        """Find similar PRs that have been reviewed"""
        if not self.vector_store:
            print("❌ Vector store not created yet!")
            return []
        
        # Perform similarity search
        similar_docs = self.vector_store.similarity_search(query, k=k)
        return similar_docs

# Create vector store for reviewer data
reviewer_vector_store = ReviewerVectorStore(embeddings)
if reviewer_loader.reviewer_data:
    reviewer_vector_store.create_embeddings(reviewer_loader.reviewer_data)
    print("🎯 Reviewer vector store ready for similarity search!")
else:
    print("❌ No reviewer data available for embedding")

c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


🔄 Creating vector embeddings for reviewer data...
✅ Created 36 vector embeddings for reviewer data
🎯 Reviewer vector store ready for similarity search!


In [6]:
import glob
from typing import List, Dict, Tuple

class SmartReviewerAssigner:
    """Enhanced system that can assign the best reviewers to PRs based on reviewer profiles and past review patterns"""
    
    def __init__(self, llm_model, vector_store, profiles_directory: str = "Reviewers Profiles"):
        self.llm = llm_model
        self.vector_store = vector_store
        self.profiles_directory = profiles_directory
        self.reviewer_profiles = {}
        self.load_reviewer_profiles()
        
        # Prompt for analyzing PR requirements
        self.analysis_prompt = PromptTemplate(
            input_variables=["pr_content"],
            template="""You are an expert technical analyst. Analyze this Pull Request and identify the key technical skills and expertise areas needed to review it effectively.

PULL REQUEST TO ANALYZE:
{pr_content}

Based on this PR, identify:
1. **Primary Technical Skills Needed**: What specific technologies, frameworks, or languages are involved?
2. **Review Expertise Areas Required**: What domain knowledge is needed (e.g., frontend, backend, testing, security, performance, documentation)?
3. **Complexity Level**: How complex is this change? (Low/Medium/High)
4. **Review Focus Areas**: What should the reviewer pay special attention to?
5. **Review Type Needed**: What type of review is most suitable? (Code Quality, Security, Performance, Documentation, Architecture)

Provide your analysis in this exact JSON format:
{{
    "technical_skills_needed": ["skill1", "skill2", "skill3"],
    "review_expertise_areas_needed": ["area1", "area2"],
    "complexity_level": "Low/Medium/High",
    "review_focus_areas": ["focus1", "focus2", "focus3"],
    "primary_language": "JavaScript/Python/etc",
    "frameworks_involved": ["framework1", "framework2"],
    "review_type_needed": "Code Quality/Security/Performance/Documentation/Architecture"
}}

ANALYSIS:"""
        )
        
        # Prompt for matching reviewers
        self.matching_prompt = PromptTemplate(
            input_variables=["pr_requirements", "reviewer_profiles", "similar_reviews", "pr_author"],
            template="""You are an expert at matching technical requirements with reviewer expertise and past review patterns. 

🎯 CRITICAL: Pay special attention to FREQUENCY DATA in reviewer profiles. 
Higher frequency numbers (e.g., "Express.js (10x)") indicate MORE REAL EXPERIENCE with that technology.
Prioritize reviewers with HIGH FREQUENCY in the required skills over those with low/no frequency data.

PR REQUIREMENTS:
{pr_requirements}

PR AUTHOR TO EXCLUDE: {pr_author}
⚠️  CRITICAL: The PR author ({pr_author}) must NEVER be assigned as a reviewer. Always exclude them from recommendations.

AVAILABLE REVIEWERS (excluding PR author):
{reviewer_profiles}

SIMILAR PAST REVIEWS:
{similar_reviews}

🎯 ASSIGNMENT STRATEGY - Use frequency data to make better decisions:
1. **Frequency Matching**: Prioritize reviewers with HIGH frequency numbers in required skills
   - "Express.js (10x)" is better than "Express.js (2x)" 
   - "Node.js (15x)" indicates extensive real-world experience
2. **Experience Level**: Match reviewer experience to PR complexity
3. **Skill Relevance**: Look for direct matches in required technologies
4. **Review Activity**: Consider overall review engagement
5. **⚠️ EXCLUDE PR AUTHOR**: Never recommend the PR author ({pr_author})

Provide your recommendation in this exact JSON format with frequency-based reasoning:
{{
    "recommended_reviewers": [ 
        {{
            "reviewer_name": "reviewer1",
            "match_score": 0.95,
            "reasoning": "High frequency in required skills: Express.js(10x), Node.js(8x). Strong experience match.",
            "strengths_alignment": ["Express.js(10x)", "Node.js(8x)", "Backend Development"],
            "review_experience": "Description emphasizing frequency-based expertise",
            "potential_concerns": "Any concerns or gaps"
        }}
    ],
    "assignment_confidence": "High/Medium/Low",
    "assignment_reasoning": "Overall reasoning emphasizing how frequency data influenced the decision"
}}

RECOMMENDATION:"""
        )
        
        self.analysis_chain = LLMChain(llm=self.llm, prompt=self.analysis_prompt)
        self.matching_chain = LLMChain(llm=self.llm, prompt=self.matching_prompt)
    
    def load_reviewer_profiles(self):
        """Load all reviewer profiles from JSON files"""
        profile_files = glob.glob(f"{self.profiles_directory}/*.json")
        
        print(f"🔍 Loading reviewer profiles from {self.profiles_directory}/...")
        
        for file_path in profile_files:
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    profile_data = json.load(f)
                
                reviewer_name = profile_data.get('reviewer_name')
                if reviewer_name:
                    self.reviewer_profiles[reviewer_name] = profile_data
                
            except Exception as e:
                print(f"❌ Error loading profile {file_path}: {e}")
        
        print(f"✅ Loaded {len(self.reviewer_profiles)} reviewer profiles")
        print(f"👥 Available reviewers: {', '.join(self.reviewer_profiles.keys())}")
    
    def display_reviewer_details(self, reviewer_name: str):
        """Display detailed information about a specific reviewer with frequency data"""
        if reviewer_name not in self.reviewer_profiles:
            print(f"❌ Reviewer '{reviewer_name}' not found")
            return
        
        profile_data = self.reviewer_profiles[reviewer_name]
        stats = profile_data.get('stats', {})
        profile = profile_data.get('profile', {})
        
        print(f"\n👨‍💻 REVIEWER PROFILE: {reviewer_name}")
        print("=" * 50)
        print(f"Experience Level: {profile.get('experience_level', 'Unknown')}")
        print(f"Languages: {', '.join(profile.get('programming_languages', []))}")
        print(f"Primary Skills: {', '.join(profile.get('primary_skills', []))}")
        print(f"Total Reviews: {stats.get('total_reviews', 0)}")
        print(f"Total Comments: {stats.get('total_comments', 0)}")
        print(f"Total PRs Reviewed: {stats.get('total_prs', 0)}")
        
        # Show top skills by frequency - extract from JavaScript skill matrix
        if 'javascript_skill_matrix' in profile:
            print("\nTop Skills by Experience:")
            js_matrix = profile['javascript_skill_matrix']
            skill_freq_pairs = []
            
            # Extract skills and frequencies from the matrix
            for category, skills in js_matrix.items():
                if skills:
                    for skill in skills:
                        # Extract frequency from skill string (e.g., "Express.js, frequency: 10")
                        if 'frequency:' in skill:
                            parts = skill.split(', frequency:')
                            if len(parts) == 2:
                                skill_name = parts[0].strip()
                                freq = int(parts[1].strip())
                                skill_freq_pairs.append((skill_name, freq))
            
            # Sort by frequency and display top 5
            sorted_skills = sorted(skill_freq_pairs, key=lambda x: x[1], reverse=True)
            for skill, freq in sorted_skills[:5]:
                print(f"  • {skill}: {freq} PRs")
        
        # Show JavaScript expertise matrix
        if 'javascript_skill_matrix' in profile:
            print("\nJavaScript Expertise Areas:")
            js_matrix = profile['javascript_skill_matrix']
            for category, skills in js_matrix.items():
                if skills:
                    # Clean category name (remove frequency info)
                    clean_category = category.split(', frequency:')[0]
                    print(f"  {clean_category}:")
                    for skill in skills[:3]:  # Top 3 in each category
                        # Clean skill name (remove frequency info)
                        clean_skill = skill.split(', frequency:')[0]
                        freq_info = skill.split(', frequency:')[1] if ', frequency:' in skill else '0'
                        print(f"    - {clean_skill} ({freq_info}x)")
        
        print(f"\nSummary: {profile.get('summary', 'No summary available')}")
        print("=" * 50)
    
    def analyze_pr_requirements(self, pr_data: Dict) -> Dict:
        """Analyze what technical skills and expertise this PR requires for review"""
        
        # Get PR summary
        pr_content = reviewer_loader.get_pr_summary(pr_data)
        
        try:
            # Analyze PR requirements
            analysis_result = self.analysis_chain.run(
                pr_content=pr_content
            )
            
            # Try to parse JSON response
            import re
            json_match = re.search(r'\{.*\}', analysis_result, re.DOTALL)
            if json_match:
                requirements = json.loads(json_match.group())
                return requirements
            else:
                print("⚠️ Could not parse PR analysis JSON, using fallback")
                return self._fallback_analysis(pr_data)
                
        except Exception as e:
            print(f"❌ Error analyzing PR requirements: {e}")
            return self._fallback_analysis(pr_data)
    
    def _fallback_analysis(self, pr_data: Dict) -> Dict:
        """Fallback analysis based on PR metadata"""
        labels = pr_data.get('labels', [])
        title = pr_data.get('title', '').lower()
        description = pr_data.get('description', '').lower()
        
        # Determine primary language (default to JavaScript for Express repo)
        primary_lang = "JavaScript"
        
        # Determine expertise areas based on content
        expertise_areas = ["General Development"]
        if any(term in title + description for term in ['test', 'spec']):
            expertise_areas.append("Testing")
        if any(term in title + description for term in ['doc', 'readme']):
            expertise_areas.append("Documentation")
        if any(term in title + description for term in ['security', 'vulnerability']):
            expertise_areas.append("Security")
        if any(term in title + description for term in ['performance', 'optimization']):
            expertise_areas.append("Performance")
        
        return {
            "technical_skills_needed": [primary_lang, "Node.js", "Express.js"],
            "review_expertise_areas_needed": expertise_areas,
            "complexity_level": "Medium",
            "review_focus_areas": ["Code Quality", "Functionality"],
            "primary_language": primary_lang,
            "frameworks_involved": ["Express.js", "Node.js"],
            "review_type_needed": "Code Quality"
        }
    
    def get_similar_reviews(self, pr_requirements: Dict, k: int = 3) -> str:
        """Get similar past reviews to help with reviewer matching"""
        try:
            # Create query from requirements
            query_parts = []
            query_parts.extend(pr_requirements.get('technical_skills_needed', []))
            query_parts.extend(pr_requirements.get('review_expertise_areas_needed', []))
            query_parts.append(pr_requirements.get('primary_language', ''))
            
            query = ' '.join(query_parts)
            
            # Find similar reviewed PRs
            similar_docs = self.vector_store.find_similar_reviewed_prs(query, k=k)
            
            similar_reviews_summary = ""
            for i, doc in enumerate(similar_docs, 1):
                metadata = doc.metadata
                reviewers = ', '.join(metadata.get('reviewers', []))
                similar_reviews_summary += f"""
SIMILAR REVIEW {i}:
PR #{metadata.get('pr_number')}: {metadata.get('title', 'No title')}
Reviewers: {reviewers}
Review Count: {metadata.get('review_count', 0)}
Content: {doc.page_content[:300]}...

"""
            
            return similar_reviews_summary
            
        except Exception as e:
            print(f"❌ Error getting similar reviews: {e}")
            return "No similar reviews found"
    
    def find_best_reviewers(self, pr_requirements: Dict, pr_data: Dict, top_k: int = 3) -> Dict:
        """Find the best reviewers for this PR (excluding the PR author)"""
        
        # 🚨 CRITICAL: Get PR author to exclude
        pr_author = pr_data.get('author', {}).get('username', 'Unknown') if pr_data.get('author') else 'Unknown'
        print(f"🚨 PR Author to exclude: {pr_author}")
        
        # Filter out the PR author from available reviewers
        available_reviewers = {name: profile for name, profile in self.reviewer_profiles.items() 
                             if name.lower() != pr_author.lower()}
        
        print(f"👥 Available reviewers (excluding author): {len(available_reviewers)} out of {len(self.reviewer_profiles)}")
        
        if not available_reviewers:
            return {
                "recommended_reviewers": [],
                "assignment_confidence": "Low",
                "assignment_reasoning": "No available reviewers after excluding PR author"
            }
        
        # Format reviewer profiles for the LLM (excluding PR author)
        profiles_summary = ""
        for reviewer_name, profile_data in available_reviewers.items():
            stats = profile_data.get('stats', {})
            profile = profile_data.get('profile', {})
            
            # Extract top skills from JavaScript skill matrix if available
            top_skills = []
            if 'javascript_skill_matrix' in profile:
                js_matrix = profile['javascript_skill_matrix']
                skill_freq_pairs = []
                
                # Extract skills and frequencies from the matrix
                for category, skills in js_matrix.items():
                    if skills:
                        for skill in skills:
                            if 'frequency:' in skill:
                                parts = skill.split(', frequency:')
                                if len(parts) == 2:
                                    skill_name = parts[0].strip()
                                    freq = int(parts[1].strip())
                                    skill_freq_pairs.append((skill_name, freq))
                
                # Sort by frequency and get top 5
                sorted_skills = sorted(skill_freq_pairs, key=lambda x: x[1], reverse=True)
                top_skills = [f"{skill} ({freq}x)" for skill, freq in sorted_skills[:5]]
            
            # Fallback to primary skills if no frequency data
            if not top_skills:
                top_skills = profile.get('primary_skills', [])
            
            # Get JavaScript-specific expertise if available
            js_expertise = ""
            if 'javascript_skill_matrix' in profile:
                js_matrix = profile['javascript_skill_matrix']
                # Look for framework expertise (clean the category names)
                framework_expertise = []
                backend_skills = []
                
                for category, skills in js_matrix.items():
                    clean_category = category.split(', frequency:')[0].lower()
                    if 'framework' in clean_category or 'library' in clean_category:
                        framework_expertise = [skill.split(', frequency:')[0] for skill in skills]
                    elif 'backend' in clean_category:
                        backend_skills = [skill.split(', frequency:')[0] for skill in skills]
                
                if framework_expertise or backend_skills:
                    js_expertise = f"\n- JavaScript Expertise: {', '.join(framework_expertise[:3] + backend_skills[:3])}"
            
            # Get a sample of review details
            review_details = profile_data.get('stats', {}).get('review_details', [])
            
            sample_reviews = ""
            for review in review_details[:3]:  # First 3 reviews as examples
                sample_reviews += f"\n    - PR #{review.get('pr_number')}: {review.get('review_state')} - {review.get('review_body', 'No comment')[:100]}..."
            
            profiles_summary += f"""
REVIEWER: {reviewer_name}
- Experience Level: {profile.get('experience_level', 'Unknown')}
- Total Reviews: {stats.get('total_reviews', 0)}
- Total Comments: {stats.get('total_comments', 0)}
- Total PRs Reviewed: {stats.get('total_prs', 0)}
- Programming Languages: {', '.join(profile.get('programming_languages', []))}
- Top Skills (by frequency): {', '.join(top_skills)}
- Primary Skills: {', '.join(profile.get('primary_skills', []))}{js_expertise}
- Repositories: {', '.join(stats.get('repos', []))}
- Summary: {profile.get('summary', 'No summary available')[:200]}...
- Sample Review Patterns:{sample_reviews}

"""
        
        # Get similar past reviews
        similar_reviews = self.get_similar_reviews(pr_requirements)
        
        try:
            # Get matching recommendations
            matching_result = self.matching_chain.run(
                pr_requirements=json.dumps(pr_requirements, indent=2),
                reviewer_profiles=profiles_summary,
                similar_reviews=similar_reviews,
                pr_author=pr_author
            )
            
            # Try to parse JSON response
            import re
            json_match = re.search(r'\{.*\}', matching_result, re.DOTALL)
            if json_match:
                recommendations = json.loads(json_match.group())
                
                # 🚨 DOUBLE CHECK: Filter out PR author from recommendations
                filtered_recommendations = []
                for rec in recommendations.get('recommended_reviewers', []):
                    if rec.get('reviewer_name', '').lower() != pr_author.lower():
                        filtered_recommendations.append(rec)
                    else:
                        print(f"🚨 BLOCKED: Prevented PR author {pr_author} from being recommended!")
                
                recommendations['recommended_reviewers'] = filtered_recommendations
                return recommendations
            else:
                print("⚠️ Could not parse matching JSON, using fallback")
                return self._fallback_matching(pr_requirements, pr_author)
                
        except Exception as e:
            print(f"❌ Error finding best reviewers: {e}")
            return self._fallback_matching(pr_requirements, pr_author)
    
    def _fallback_matching(self, pr_requirements: Dict, pr_author: str) -> Dict:
        """Enhanced fallback matching using skill frequency data from JavaScript matrix (excluding PR author)"""
        
        # 🚨 Filter out PR author from available reviewers
        available_reviewers = {name: profile for name, profile in self.reviewer_profiles.items() 
                             if name.lower() != pr_author.lower()}
        
        if not available_reviewers:
            return {
                "recommended_reviewers": [],
                "assignment_confidence": "Low",
                "assignment_reasoning": f"No available reviewers after excluding PR author ({pr_author})"
            }
        
        primary_lang = pr_requirements.get('primary_language', 'JavaScript')
        needed_skills = pr_requirements.get('technical_skills_needed', [])
        needed_frameworks = pr_requirements.get('frameworks_involved', [])
        
        # Enhanced scoring based on skill frequency, expertise, and review activity
        reviewer_scores = []
        for reviewer_name, profile_data in available_reviewers.items():
            score = 0.0
            stats = profile_data.get('stats', {})
            profile = profile_data.get('profile', {})
            
            # Get all available skills
            reviewer_skills = profile.get('primary_skills', []) + profile.get('programming_languages', [])
            
            # Extract skill frequencies from JavaScript skill matrix
            skill_frequencies = {}
            if 'javascript_skill_matrix' in profile:
                js_matrix = profile['javascript_skill_matrix']
                for category, skills in js_matrix.items():
                    if skills:
                        for skill in skills:
                            if 'frequency:' in skill:
                                parts = skill.split(', frequency:')
                                if len(parts) == 2:
                                    skill_name = parts[0].strip()
                                    freq = int(parts[1].strip())
                                    skill_frequencies[skill_name] = freq
            
            # Language match with frequency bonus
            if primary_lang in profile.get('programming_languages', []):
                base_score = 0.3
                # Bonus for high frequency of that language
                if primary_lang in skill_frequencies:
                    frequency_bonus = min(skill_frequencies[primary_lang] / 15, 0.2)  # Max 0.2 bonus
                    score += base_score + frequency_bonus
                else:
                    score += base_score
            
            # Skill overlap with frequency weighting
            for skill in needed_skills:
                if skill in skill_frequencies:
                    # Weight by skill frequency (more experience = higher score)
                    frequency_weight = min(skill_frequencies[skill] / 8, 0.25)  # Max 0.25 per skill
                    score += frequency_weight
                elif skill in reviewer_skills:
                    score += 0.08  # Basic skill match
            
            # Framework expertise bonus
            if 'javascript_skill_matrix' in profile:
                js_matrix = profile['javascript_skill_matrix']
                # Look for framework expertise in any category containing "framework" or "library"
                for category, skills in js_matrix.items():
                    clean_category = category.split(', frequency:')[0].lower()
                    if 'framework' in clean_category or 'library' in clean_category:
                        framework_expertise = [skill.split(', frequency:')[0] for skill in skills]
                        for framework in needed_frameworks:
                            if any(framework.lower() in expertise.lower() for expertise in framework_expertise):
                                score += 0.15
            
            # Experience level bonus
            experience_level = profile.get('experience_level', 'Junior')
            complexity = pr_requirements.get('complexity_level', 'Medium')
            if (complexity == 'High' and experience_level in ['Senior', 'Expert']) or \
               (complexity == 'Medium' and experience_level in ['Mid', 'Senior', 'Expert']) or \
               (complexity == 'Low'):
                score += 0.1
            
            # Review activity scoring (reduced weight to balance with skill scoring)
            total_reviews = stats.get('total_reviews', 0)
            if total_reviews > 0:
                score += min(total_reviews / 100, 0.2)  # Max 0.2 for review count
            
            # Score based on PR count
            total_prs = stats.get('total_prs', 0)
            if total_prs > 0:
                score += min(total_prs / 60, 0.15)  # Max 0.15 for PR count
            
            # Score based on comment engagement
            total_comments = stats.get('total_comments', 0)
            if total_comments > 0:
                score += min(total_comments / 40, 0.1)  # Max 0.1 for comments
            
            reviewer_scores.append((reviewer_name, score))
        
        # Sort by score and take top 3
        reviewer_scores.sort(key=lambda x: x[1], reverse=True)
        top_3 = reviewer_scores[:3]
        
        recommendations = []
        for i, (reviewer_name, score) in enumerate(top_3):
            stats = available_reviewers[reviewer_name].get('stats', {})
            profile = available_reviewers[reviewer_name].get('profile', {})
            
            # Get top skills for this reviewer
            top_skills = []
            if 'javascript_skill_matrix' in profile:
                js_matrix = profile['javascript_skill_matrix']
                skill_freq_pairs = []
                
                # Extract skills and frequencies from the matrix
                for category, skills in js_matrix.items():
                    if skills:
                        for skill in skills:
                            if 'frequency:' in skill:
                                parts = skill.split(', frequency:')
                                if len(parts) == 2:
                                    skill_name = parts[0].strip()
                                    freq = int(parts[1].strip())
                                    skill_freq_pairs.append((skill_name, freq))
                
                # Sort by frequency and get top 3
                sorted_skills = sorted(skill_freq_pairs, key=lambda x: x[1], reverse=True)
                top_skills = [skill for skill, freq in sorted_skills[:3]]
            
            # Fallback to primary skills
            if not top_skills:
                top_skills = profile.get('primary_skills', [])[:3]
            
            # Generate more detailed reasoning
            reasoning_parts = []
            if primary_lang in profile.get('programming_languages', []):
                reasoning_parts.append(f"Strong {primary_lang} experience")
            
            # Check for skill matches in the extracted frequencies
            skill_matches = []
            if 'javascript_skill_matrix' in profile:
                js_matrix = profile['javascript_skill_matrix']
                for category, skills in js_matrix.items():
                    for skill in skills:
                        skill_name = skill.split(', frequency:')[0].strip()
                        if skill_name in needed_skills:
                            skill_matches.append(skill_name)
            
            if skill_matches:
                reasoning_parts.append(f"Experienced in {', '.join(skill_matches[:2])}")
            
            reasoning_parts.append(f"Active reviewer with {stats.get('total_reviews', 0)} reviews")
            
            if not reasoning_parts:
                reasoning_parts.append("General review skills match")
            
            recommendations.append({
                "reviewer_name": reviewer_name,
                "match_score": min(score, 1.0),
                "reasoning": "; ".join(reasoning_parts),
                "strengths_alignment": top_skills,
                "review_experience": f"Has reviewed {stats.get('total_prs', 0)} PRs with {stats.get('total_comments', 0)} comments",
                "potential_concerns": "Limited specific expertise" if score < 0.5 else "None identified"
            })
        
        confidence = "High" if top_3[0][1] > 0.7 else ("Medium" if top_3[0][1] > 0.4 else "Low")
        
        return {
            "recommended_reviewers": recommendations,
            "assignment_confidence": confidence,
            "assignment_reasoning": f"Enhanced matching using skill frequency and review activity data (PR author {pr_author} excluded)"
        }
    
    def smart_reviewer_assignment(self, pr_data: Dict) -> Tuple[str, Dict]:
        """Complete workflow: analyze PR, find best reviewers"""
        
        print(f"🔍 Analyzing PR #{pr_data.get('pr_number')}: {pr_data.get('title')}")
        print("=" * 80)
        
        # Step 1: Analyze PR requirements
        print("📊 Step 1: Analyzing PR review requirements...")
        pr_requirements = self.analyze_pr_requirements(pr_data)
        
        print(f"✅ Required Skills: {', '.join(pr_requirements.get('technical_skills_needed', []))}")
        print(f"✅ Review Areas: {', '.join(pr_requirements.get('review_expertise_areas_needed', []))}")
        print(f"✅ Complexity: {pr_requirements.get('complexity_level', 'Unknown')}")
        print(f"✅ Review Type: {pr_requirements.get('review_type_needed', 'Unknown')}")
        
        # Step 2: Find best reviewers (excluding author)
        print("\n👥 Step 2: Finding best-matched reviewers...")
        reviewer_recommendations = self.find_best_reviewers(pr_requirements, pr_data)
        
        print("🎯 Recommended Reviewers:")
        for i, rec in enumerate(reviewer_recommendations.get('recommended_reviewers', []), 1):
            print(f"  {i}. {rec['reviewer_name']} (Score: {rec['match_score']:.2f})")
            print(f"     Reasoning: {rec['reasoning']}")
            print(f"     Experience: {rec.get('review_experience', 'Not specified')}")
            print(f"     Strengths: {', '.join(rec.get('strengths_alignment', []))}")
            if rec.get('potential_concerns'):
                print(f"     Concerns: {rec['potential_concerns']}")
            print()
        
        # Combine results
        pr_author = pr_data.get('author', {}).get('username', 'Unknown') if pr_data.get('author') else 'Unknown'
        assignment_summary = f"""
🎯 SMART REVIEWER ASSIGNMENT SUMMARY
{'=' * 50}

📋 PR REVIEW REQUIREMENTS:
- Technical Skills Needed: {', '.join(pr_requirements.get('technical_skills_needed', []))}
- Review Expertise Areas: {', '.join(pr_requirements.get('review_expertise_areas_needed', []))}
- Complexity Level: {pr_requirements.get('complexity_level', 'Unknown')}
- Primary Language: {pr_requirements.get('primary_language', 'Unknown')}
- Review Type Needed: {pr_requirements.get('review_type_needed', 'Unknown')}

🚨 PR AUTHOR EXCLUDED: {pr_author}

👥 RECOMMENDED REVIEWERS:
"""
        
        for i, rec in enumerate(reviewer_recommendations.get('recommended_reviewers', []), 1):
            assignment_summary += f"""
{i}. **{rec['reviewer_name']}** (Match Score: {rec['match_score']:.2f})
   - Reasoning: {rec['reasoning']}
   - Review Experience: {rec.get('review_experience', 'Not specified')}
   - Key Strengths: {', '.join(rec.get('strengths_alignment', [])[:3])}
   - Concerns: {rec.get('potential_concerns', 'None identified')}
"""
        
        assignment_summary += f"""
🔍 Assignment Confidence: {reviewer_recommendations.get('assignment_confidence', 'Unknown')}
📝 Assignment Reasoning: {reviewer_recommendations.get('assignment_reasoning', 'Not provided')}

{'=' * 50}
"""
        
        return assignment_summary, {
            'pr_requirements': pr_requirements,
            'reviewer_recommendations': reviewer_recommendations
        }

# Create the smart reviewer assigner
print("🚀 Creating Smart Reviewer Assigner...")
smart_reviewer_assigner = SmartReviewerAssigner(llm, reviewer_vector_store, "Reviewers Profiles")
print("✅ Smart Reviewer Assigner ready!")

🚀 Creating Smart Reviewer Assigner...
🔍 Loading reviewer profiles from Reviewers Profiles/...
✅ Loaded 30 reviewer profiles
👥 Available reviewers: 0xflotus, aashutoshrathi, aminya, armano2, blikblum, canscaw28, Cassieminkus1, danimajo, ekkis, falsyvalues, Gauravms2143, HaseebIjaz, Hovakimyan, imadx, JakeerC, jdalton, jonchurch, Kirill89, mohamadaminkarami, phapdinh, qiudaoermu, spencer17x, TheJaredWilcurt, tobie, Toxicable, Trott, UlisesGascon, whiteand, xgqfrms, Yatin-kathuria
✅ Smart Reviewer Assigner ready!


C:\Users\ASUS\AppData\Local\Temp\ipykernel_31372\248574614.py:92: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  self.analysis_chain = LLMChain(llm=self.llm, prompt=self.analysis_prompt)


In [8]:
# Quick Reviewer Assignment Function - for easy testing with different PRs
def quick_assign_reviewer(pr_number: int):
    """Quick function to assign reviewers to a PR by number"""
    # Find PR by number
    target_pr = None
    for pr in reviewer_loader.reviewer_data:
        if pr.get('pr_number') == pr_number:
            target_pr = pr
            break
    
    if not target_pr:
        print(f"❌ PR #{pr_number} not found in loaded reviewer data")
        return
    
    print(f"🎯 SMART REVIEWER ASSIGNMENT FOR PR #{pr_number}")
    print(f"📝 Title: {target_pr.get('title')}")
    print(f"👤 Author: {target_pr.get('author', {}).get('username', 'Unknown')}")
    print("=" * 60)
    
    # Get assignment recommendations (now properly excluding the author)
    pr_requirements = smart_reviewer_assigner.analyze_pr_requirements(target_pr)
    reviewer_recommendations = smart_reviewer_assigner.find_best_reviewers(pr_requirements, target_pr)  # 🚨 Pass PR data
    
    # Get the best reviewer (first in the list)
    recommended_reviewers = reviewer_recommendations.get('recommended_reviewers', [])
    if not recommended_reviewers:
        print("❌ No suitable reviewer found for this PR")
        return
    
    best_reviewer = recommended_reviewers[0]  # Get only the top recommendation
    
    print("🎯 ASSIGNED REVIEWER:")
    print(f"\n👨 **{best_reviewer['reviewer_name']}** (Match: {best_reviewer['match_score']:.0%})")
    print(f"💡 Why: {best_reviewer['reasoning']}")
    print(f"⭐ Strengths: {', '.join(best_reviewer.get('strengths_alignment', [])[:3])}")
    if best_reviewer.get('potential_concerns'):
        print(f"⚠️ Concerns: {best_reviewer['potential_concerns']}")
    
    print(f"\n🔍 Assignment Confidence: {reviewer_recommendations.get('assignment_confidence', 'Unknown')}")
    
    # 🚨 SAFETY CHECK: Verify author is not assigned
    pr_author = target_pr.get('author', {}).get('username', 'Unknown') if target_pr.get('author') else 'Unknown'
    assigned_reviewer = best_reviewer['reviewer_name']
    
    if assigned_reviewer.lower() == pr_author.lower():
        print("🚨🚨🚨 CRITICAL ERROR: PR AUTHOR WAS ASSIGNED AS REVIEWER!")
        print(f"Author: {pr_author}, Assigned: {assigned_reviewer}")
        return
    else:
        print(f"✅ SAFETY CHECK PASSED: Author ({pr_author}) ≠ Assigned Reviewer ({assigned_reviewer})")
    
    print("=" * 60)


quick_assign_reviewer(4355)

🎯 SMART REVIEWER ASSIGNMENT FOR PR #4355
📝 Title: Prevent prototype pollution chaining to code execution via _.template
👤 Author: alexbrasetvik


C:\Users\ASUS\AppData\Local\Temp\ipykernel_31372\248574614.py:184: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  analysis_result = self.analysis_chain.run(


🚨 PR Author to exclude: alexbrasetvik
👥 Available reviewers (excluding author): 30 out of 30


c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


🎯 ASSIGNED REVIEWER:

👨 **jdalton** (Match: 95%)
💡 Why: High frequency in required skills: lodash(8x), Node.js(2x), Security Best Practices. Strong experience match with a focus on security and functional programming.
⭐ Strengths: lodash(8x), Node.js(2x), Security Best Practices
⚠️ Concerns: None significant

🔍 Assignment Confidence: High
✅ SAFETY CHECK PASSED: Author (alexbrasetvik) ≠ Assigned Reviewer (jdalton)
